## Import bibliotek

In [1]:
import requests
from bs4 import BeautifulSoup

## Pobranie z serwisu [Ceneo.pl](https://www.ceneo.pl) opinii o wybranym produkcie

In [18]:
headers = {
    "Cookie": "__RequestVerificationToken=7Q2UUZePLkDZ6vrbfL1CdAuWXpMYSONA1ieDuRODLj149LKk1rqTXy64_5QRiTAP-vpBwy0rI_83RigfRM1cvdVoKUD3wazuvJNeUSEis4g1",
    "User-Agent":"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Host":"www.ceneo.pl"
}

In [9]:
url = "https://www.ceneo.pl/47293506#tab=reviews"
response = requests.get(url, headers=headers)
print(response.status_code)

200


## Parsowanie strony z opiniami o produkcie

In [3]:
page_dom = BeautifulSoup(response.text, 'html.parser')

In [10]:
opinions = page_dom.select("div.js_product-review:not(.user-post--highlight)")
print(len(opinions))
print(type(opinions))

10
<class 'bs4.element.ResultSet'>


In [11]:
opinion = page_dom.select_one("div.js_product-review:not(.user-post--highlight)")
print(type(opinion))

<class 'bs4.element.Tag'>


In [6]:
opinion = opinions.pop(0)
print(type(opinion))

<class 'bs4.element.Tag'>


## Analiza struktury pojedynczej opinii

|składowa|nazwa|selektor|
|--------|-----|--------|
|opinia|opinion|div.js_product-review:not(.user-post--highlight)|
|identyfikator|opinion_id|[data-entry-id]|
|autor|author|span.user-post__author-name|
|treść|content|div.user-post__text|
|ocena|score|span.user-post__score-count|
|rekomendacja|recomendation|span.user-post__author-recomendation > em|
|lista zalet|pros|div.review-feature__item--positive|
|lista wad|cons|div.review-feature__item--negative|
|dla ilu przydatna|useful|button.vote-yes > span|
|dla ilu nieprzydatna|unuseful|button.vote-no > span|
|data zamieszczenia|publish_date|span.user-post__published > time:nth-child(1)[datetime]|
|data zakupu|purchase_date|span.user-post__published > time:nth-child(2)[datetime]|

In [13]:
all_opinions = []

for opinion in opinions:
    opinion_data = {}
    
    # identyfikator opinii
    opinion_data["id"] = opinion.get("data-entry-id")
    
    # Autor
    opinion_data["author"]=opinion.select_one("span.user-post__author-name").get_text(strip=True)
    
    # Treść opinii
    opinion_data["content"]=opinion.select_one("div.user-post__text").get_text(strip=True)
    
    # Ocena
    opinion_data["score"]=opinion.select_one("span.user-post__score-count").get_text(strip=True)
    opinion_data["score"]=float(opinion_data["score"].split("/")[0].replace(",","."))
    
    # Rekomendacja (Polecam/Nie polecam)
    try:
        opinion_data["recommendation"] = opinion.select_one("span.user-post__author-recommendation > em").get_text(strip=True)
    except AttributeError:
        opinion_data["recommendation"] = None

    # Lista zalet
    opinion_data["cons"]=[p.get_text(strip=True) for p in opinion.select("div.review-feature__item--positive")]
    
    # Wady
    opinion_data["cons"]=[p.get_text(strip=True) for p in opinion.select("div.review-feature__item--negative")]

    
    # Dla ilu osób przydatna
    opinion_data["useful"]=int(opinion.select_one("button.vote-yes > span").get_text(strip=True))
    
    # Dla ilu osób nieprzydatna
    opinion_data["useful"]=int(opinion.select_one("button.vote-no > span").get_text(strip=True))
    
    # Data zamieszczenia
    opinion_data["published_date"]=opinion.select_one("span.user-post__published > time:nth-child(1)").get("datetime")
    
    # Data zakupu / potwierdzony zakup
    try:
        opinion_data["purchase_date"]=opinion.select_one("span.user-post__published > time:nth-child(2)").get("datetime")
    except AttributeError:
        opinion_data["purchase_date"]=None
    
    all_opinions.append(opinion_data)
    
# Wyświetlenie zebranych opinii
for single_opinion in all_opinions:
    print(single_opinion)
    

{'id': '6572033', 'author': 'Paweł', 'content': 'Nie jest całkowicie bezgłośna, ale poziom dźwięku kliknięć jest dużo niższy a i jego dźwięk bardziej przyjemny i chyba mniej irytujący. Poza tym wygodna myszka, miłe w dotyku materiały. Ja jestem zadowolony i polecam :)', 'score': 5.0, 'recommendation': None, 'cons': [], 'useful': 0, 'published_date': '2018-02-08 21:32:09', 'purchase_date': '2018-02-04 18:52:38'}
{'id': '16601645', 'author': 'k...r', 'content': 'Myszka jest rewelacyjna. To jakby porównać odgłos klawiatury z czasów WIN95 z współczesną cichutką klawiaturą laptopową. I jedno i drugie będzie działało ale po co słuchać tego irytującego klikania. A pomnóż  to np. x10 osób w biurze:) \nBardzo udany zakup.', 'score': 5.0, 'recommendation': None, 'cons': [], 'useful': 0, 'published_date': '2022-10-06 11:51:00', 'purchase_date': '2022-09-30 08:22:14'}
{'id': '19734571', 'author': 'a...w', 'content': 'Generalnie myszka super. Cicha i tak jak piszą inni, trzeba się do tego przyzwycz

In [19]:
page = 1
next = True
while next:
    url = f"https://www.ceneo.pl/47293506/opinie-{page}"
    response = requests.get(url, headers=headers)
    print(f"{url} => {response.status_code} => {len(opinions)}")
    page_dom = BeautifulSoup(response.text, "html.parser")
    next = True if page_dom.select_one("button.pagination__next") else False
    page += 1

https://www.ceneo.pl/47293506/opinie-1 => 200 => 10
https://www.ceneo.pl/47293506/opinie-2 => 200 => 10
